# 05 - Industrial BTM Refinery & Multi-Energy Complex

This case study simulates a complex Behind-The-Meter (BTM) industrial chemical / refinery facility operating over a 24-hour cycle:

* **12 MW Solar PV Plant** (`PhysicsPV`)
* **40 MW Combined Cycle Gas Turbine** (`CombinedCycle_CCGT`)
* **10 MW Green Hydrogen PEM Electrolyzer** (`PEMElectrolyzer`)
* **350-bar Compressed H2 Buffer Storage** (`H2StorageTank`)
* **Tesla Megapack 2XL Battery System** (`catalog.Tesla_Megapack_2XL`)
* **Industrial Process Steam Header Demand** (`IndustrialProcessSteam`)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from energysim import world
from energysim.models import catalog
from energysim.models.renewables import PhysicsPV
from energysim.models.hydrogen import PEMElectrolyzer, H2StorageTank
from energysim.models.thermal import CombinedCycle_CCGT
from energysim.models.loads import IndustrialProcessSteam

print("Industrial components imported!")


## Step 1: Set Up the Industrial Complex Canvas


In [ ]:
sim = world(start_time=0, stop_time=86400, t_macro=900, logging=False)

# 1. Solar PV
solar = PhysicsPV(name="solar_pv", p_dc_stc_mw=12.0, inverter_p_ac_max_mw=10.0)

# 2. BESS Archetype from catalog
bess = catalog.Tesla_Megapack_2XL(name="bess", soc_init=0.5)

# 3. Combined Cycle Gas Turbine
ccgt = CombinedCycle_CCGT(name="ccgt", p_rated_mw=40.0, min_stable_gen_pct=35.0)

# 4. PEM Electrolyzer & Tank
electrolyzer = PEMElectrolyzer(name="electrolyzer", p_nominal_mw=10.0, n_cells=1800)
tank = H2StorageTank(name="h2_tank", volume_m3=100.0, p_max_bar=350.0, p_init_bar=120.0)

# 5. Steam Header
steam = IndustrialProcessSteam(name="steam_header", nominal_demand_t_hr=40.0)

# Add all components
sim.add_model("solar_pv", model=solar, outputs=["P_ac", "T_cell"])
sim.add_model("bess", model=bess, outputs=["P_actual", "SoC", "T_batt"])
sim.add_model("ccgt", model=ccgt, outputs=["P_actual", "fuel_consumption_MMBtu_hr", "CO2_emissions_t_hr"])
sim.add_model("electrolyzer", model=electrolyzer, outputs=["P_actual", "m_dot_H2_kg_hr"])
sim.add_model("h2_tank", model=tank, outputs=["pressure_bar", "mass_stored_kg"])
sim.add_model("steam_header", model=steam, outputs=["steam_demand_t_hr"])


## Step 2: Establish Diurnal Weather and Plant Dispatch Signals


In [ ]:
# Solar Irradiance
def solar_profile(t):
    hr = (t / 3600.0) % 24.0
    return 950.0 * np.sin(np.pi * (hr - 6.0) / 12.0) if 6.0 <= hr <= 18.0 else 0.0

# Ambient Temperature
def temp_profile(t):
    hr = (t / 3600.0) % 24.0
    return 20.0 + 10.0 * np.sin(np.pi * (hr - 9.0) / 12.0)

# Electrolyzer flex: soak up midday solar
def h2_dispatch(t):
    hr = (t / 3600.0) % 24.0
    return 8.5 if 9.0 <= hr <= 16.0 else 2.5

# BESS peak shaving
def bess_dispatch(t):
    hr = (t / 3600.0) % 24.0
    if 11.0 <= hr <= 14.5:
        return 1.8
    elif 18.0 <= hr <= 21.0:
        return -1.8
    return 0.0

sim.add_signal("irr_sig", signal=solar_profile)
sim.add_signal("temp_sig", signal=temp_profile)
sim.add_signal("h2_sig", signal=h2_dispatch)
sim.add_signal("bess_sig", signal=bess_dispatch)
sim.add_signal("ccgt_sig", signal=lambda t: 25.0)  # 25 MW baseload

# Connect flows
sim.add_connections({
    "irr_sig.y": "solar_pv.G_poa",
    "temp_sig.y": ("solar_pv.T_amb", "bess.T_amb", "ccgt.T_amb"),
    "h2_sig.y": "electrolyzer.P_cmd",
    "bess_sig.y": "bess.P_cmd",
    "ccgt_sig.y": "ccgt.P_cmd",
    "electrolyzer.m_dot_H2_kg_hr": "h2_tank.m_dot_in_kg_hr",
})


## Step 3: Run Co-Simulation and Analyze Dispatch


In [ ]:
sim.simulate(pbar=True)
results = sim.last_results

print("Co-simulation complete! Results summary:")
for k, df in results.items():
    print(f" - {k}: {len(df)} time-steps recorded.")
